author: loua

## Setup

In [1]:
import pandas as pd

ROOT = '../../../'
PARENT_PATH = ROOT + 'raw-data/'
GEO_PATH = PARENT_PATH + 'historical-elections/election-results/Geography.csv'
CPH_PATH = PARENT_PATH + 'election-2025/Kommunalvalg_2025_København_25-11-2025 14.54.11.csv'
FRB_PATH = PARENT_PATH + 'election-2025/Kommunalvalg_2025_Frederiksberg_25-11-2025 14.54.22.csv'
OUTPUT_PATH = ROOT + 'processed-data/elections/'
YEARS = [2009, 2013, 2017, 2021]
MUNICIPALITY_IDS = [101, 147]  # Copenhagen and Frederiksberg

## Concatenate CPH and FRB

In [2]:
cph_2025 = pd.read_csv(CPH_PATH, sep=';', encoding='utf-8')
frb_2025 = pd.read_csv(FRB_PATH, sep=';', encoding='utf-8')
cph_2025['Municipality_Source'] = 'København'
frb_2025['Municipality_Source'] = 'Frederiksberg'
df = pd.concat([cph_2025, frb_2025], ignore_index=True)

## Get votes

In [3]:
# Group by polling area and party, sum the votes
votes_by_area_party = df.groupby(['Afstemningsområde', 'Bogstavbetegnelse']).agg({
    'Stemmetal': 'sum',
    'Municipality_Source': 'first'
}).reset_index()

# Pivot to get parties as columns
votes_pivot = votes_by_area_party.pivot(index=['Afstemningsområde', 'Municipality_Source'], 
                                         columns='Bogstavbetegnelse', 
                                         values='Stemmetal').reset_index()
votes_pivot = votes_pivot.fillna(0)

# Calculate total votes
party_cols = [col for col in votes_pivot.columns if col not in ['Afstemningsområde', 'Municipality_Source']]
votes_pivot['Valid votes'] = votes_pivot[party_cols].sum(axis=1)

# Convert party columns to int if no fractional part
for col in party_cols + ['Valid votes']:
    if (votes_pivot[col] % 1 == 0).all():
        votes_pivot[col] = votes_pivot[col].astype(int)

## Merge on geodata

In [4]:
# Extract polling area number and name from 'Afstemningsområde'
votes_pivot['Afstemningsområde'] = votes_pivot['Afstemningsområde'].astype(str)
votes_pivot['area_num'] = votes_pivot['Afstemningsområde'].str.split('.', n=1).str[0].str.strip()
votes_pivot['area_name'] = votes_pivot['Afstemningsområde'].str.split('.', n=1).str[1].str.strip()

geo_df = pd.read_csv(GEO_PATH, sep=';', encoding='utf-8', quotechar='"', dtype=str)
geo_df.columns = geo_df.columns.str.strip().str.strip('"')
geo_df = geo_df[geo_df['KommuneNr'].isin(['101', '147'])].copy()

# Create a mapping from Geography
geo_df['area_num_from_id'] = geo_df['Valgsted Id'].astype(str).str[-2:].astype(int).astype(str)

# Try matching with both number and name first
merged = votes_pivot.merge(
    geo_df[['Valgsted Id', 'Valgsted navn', 'KommuneNr', 'Kreds Nr', 'Kreds navn', 'Kommune navn', 'area_num_from_id']],
    left_on=['area_num', 'area_name'],
    right_on=['area_num_from_id', 'Valgsted navn'],
    how='left'
)

# Mark matched rows
merged['Matched'] = merged['Valgsted Id'].notna().astype(int)

# Handle unmatched rows
unmatched_mask = merged['Valgsted Id'].isna()
unmatched_count = unmatched_mask.sum()

if unmatched_count > 0:
    print(f"Handling {unmatched_count} unmatched areas")
    
    for idx in merged[unmatched_mask].index:
        area_num = str(merged.loc[idx, 'area_num'])
        area_name = str(merged.loc[idx, 'area_name'])
        muni_source = merged.loc[idx, 'Municipality_Source']
        
        # Determine municipality ID from Kreds name pattern or source
        if 'Kreds' in area_name and muni_source == 'Frederiksberg':
            muni_id = '147'
            muni_name = 'Frederiksberg'
        else:
            muni_id = '101'
            muni_name = 'København'
        
        # Generate Valgsted Id: {muni_id}0{area_num padded to 2 digits}
        area_num_padded = area_num.zfill(2)
        valgsted_id = f"{muni_id}0{area_num_padded}"
        
        # Extract DistrictNo from area_name
        first_part = area_name.split('.')[0].strip()
        if first_part.isdigit():
            district_no = first_part
        else:
            district_no = area_num  # fallback
        
        # Look up District name from Geography
        district_match = geo_df[(geo_df['KommuneNr'] == muni_id) & 
                                (geo_df['Kreds Nr'] == district_no)]

        if len(district_match) > 0:
            district_name = district_match.iloc[0]['Kreds navn']
        else:
            district_name = ''  # Could not find district
        
        # Fill in the values
        merged.loc[idx, 'Valgsted Id'] = valgsted_id
        merged.loc[idx, 'Valgsted navn'] = area_name
        merged.loc[idx, 'KommuneNr'] = muni_id
        merged.loc[idx, 'Kreds Nr'] = district_no
        merged.loc[idx, 'Kreds navn'] = district_name
        merged.loc[idx, 'Kommune navn'] = muni_name

Handling 11 unmatched areas


## Select and translate columns

In [5]:
# Rename columns to English
merged = merged.rename(columns={
    'Valgsted Id': 'Gruppe',
    'Valgsted navn': 'Name',
    'Kreds Nr': 'DistrictNo',
    'Kreds navn': 'District',
    'Kommune navn': 'Municipality'
})

# Add PollingAreaID column (same as Gruppe)
merged['PollingAreaID'] = merged['Gruppe']

# Convert columns to string
str_cols = ['Gruppe', 'PollingAreaID', 'Name', 'DistrictNo', 'District', 'Municipality']
for col in str_cols:
    merged[col] = merged[col].fillna('').astype(str).str.replace(r'\.0$', '', regex=True)

# Select and reorder columns
final_column_order = ['Gruppe', 'PollingAreaID', 'Name', 'DistrictNo', 'District', 'Municipality', 'Matched', 'Valid votes'] + party_cols
df_2025 = merged[final_column_order].copy()

# Sort by Gruppe
df_2025 = df_2025.sort_values('Gruppe').reset_index(drop=True)


In [6]:
df_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 32 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Gruppe         66 non-null     object
 1   PollingAreaID  66 non-null     object
 2   Name           66 non-null     object
 3   DistrictNo     66 non-null     object
 4   District       66 non-null     object
 5   Municipality   66 non-null     object
 6   Matched        66 non-null     int64 
 7   Valid votes    66 non-null     int64 
 8   A              66 non-null     int64 
 9   B              66 non-null     int64 
 10  C              66 non-null     int64 
 11  D              66 non-null     int64 
 12  E              66 non-null     int64 
 13  F              66 non-null     int64 
 14  I              66 non-null     int64 
 15  J              66 non-null     int64 
 16  K              66 non-null     int64 
 17  L              66 non-null     int64 
 18  M              66 non-null     i

In [7]:
df_2025.head()

,Gruppe,PollingAreaID,Name,DistrictNo,District,Municipality,Matched,Valid votes,A,B,...,Q,R,T,U,V,Y,Z,Å,Æ,Ø
0,101001,101001,1. Østerbro,1,1. Østerbro,København,1,6868,1073,799,...,43,11,8,2,506,5,0,227,27,1097
1,101002,101002,1. Nord,1,1. Østerbro,København,1,5074,674,524,...,31,7,10,2,453,2,1,183,22,696
2,101003,101003,1. Syd,1,1. Østerbro,København,1,6515,852,787,...,37,12,12,1,458,3,0,295,20,1180
3,101005,101005,1. Vest,1,1. Østerbro,København,1,8276,1176,828,...,81,15,23,3,561,1,1,401,40,1539
4,101006,101006,1. Nordvest,1,1. Østerbro,København,1,7600,1056,875,...,85,14,20,2,502,3,3,333,44,1387


## Save to CSV

In [8]:
output_file = OUTPUT_PATH + 'absolute_2025.csv'
df_2025.to_csv(output_file, index=False)
print(f"\nSaved {output_file}")
print(f"Shape: {df_2025.shape}")
print(f"Matched: {df_2025['Matched'].sum()}, Unmatched: {(1-df_2025['Matched']).sum()}")
print(f"\nFirst few rows:")
print(df_2025.head())


Saved ../../../processed-data/elections/absolute_2025.csv
Shape: (66, 32)
Matched: 55, Unmatched: 11

First few rows:
   Gruppe PollingAreaID         Name DistrictNo     District Municipality  \
0  101001        101001  1. Østerbro          1  1. Østerbro    København   
1  101002        101002      1. Nord          1  1. Østerbro    København   
2  101003        101003       1. Syd          1  1. Østerbro    København   
3  101005        101005      1. Vest          1  1. Østerbro    København   
4  101006        101006  1. Nordvest          1  1. Østerbro    København   

   Matched  Valid votes     A    B  ...   Q   R   T  U    V  Y  Z    Å   Æ  \
0        1         6868  1073  799  ...  43  11   8  2  506  5  0  227  27   
1        1         5074   674  524  ...  31   7  10  2  453  2  1  183  22   
2        1         6515   852  787  ...  37  12  12  1  458  3  0  295  20   
3        1         8276  1176  828  ...  81  15  23  3  561  1  1  401  40   
4        1         7600  105

## Convert to percentages and save as new CSV

In [9]:
# Convert party columns to percentage of 'Valid votes'
df_2025_rel = df_2025.copy()
df_2025_rel[party_cols] = df_2025_rel[party_cols].div(df_2025_rel['Valid votes'], axis=0) * 100

# Save to CSV
output_file = OUTPUT_PATH + 'relative_2025.csv'
df_2025_rel.to_csv(output_file, index=False)
print(f"Saved {output_file}")

Saved ../../../processed-data/elections/relative_2025.csv
